In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Initialization

Every network we have trained began from random weights, and we have mostly
let the framework draw them. that section showed that the
draw matters: if each layer scales the signal up or down a little, a deep
stack scales it up or down a lot, and the network does not train.

That section also gave the cure for a single layer. Draw each weight with
standard deviation $g/\sqrt{n_\textrm{in}}$, where $n_\textrm{in}$ is the
number of inputs of the layer and the *gain* $g$ depends on what follows it:
$g = \sqrt{2}$ before a ReLU, which is He initialization
(that section), and $g = 1$ when no nonlinearity follows. The
layer then passes on the mean square of its input unchanged.

One might hope that a rule for each layer settles the matter. It does not. In
a residual network the activations grow even though every layer follows the
rule. Each residual block adds the output of a small network, its body $f$
(that section), to its input. If the body passes on the
mean square of its input, and its output is uncorrelated with that input, the
sum has twice the mean square of the input. After 32 blocks the standard
deviation of the activations has grown by a factor of
$2^{16} \approx 65{,}000$, before training has started.

The right initialization therefore depends on how the layers are connected,
not only on each layer, and we need a way to apply a chosen rule to chosen
parameters of a model. Most of this section is practice, and its first half
can be read quickly: what the framework does when we choose nothing, how to
apply He initialization to a deep stack of layers, how to write an
initializer of our own, and how to edit parameters by hand. The second half
is the part to read slowly. There we repair the residual network, by scaling
down or zeroing the last layer of each body.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## Library Defaults

A framework initializes a layer when we do not, and none of the four
frameworks of this book defaults to He initialization
(that section). A statement like this is easy to check,
so let us check it. We draw a layer with 512 inputs and 256 outputs and
compare its weights with the documented rule. Where the rule is a uniform
distribution on $[-b, b]$, the predicted standard deviation is $b/\sqrt{3}$.

Keras creates the kernel at build time, when the layer learns its input shape
(that section), so we call `build` ourselves. It draws Xavier
uniform, which Keras calls Glorot uniform after the first author of
@Glorot.Bengio.2010, and starts the bias at zero:

In [ ]:
layer = tf.keras.layers.Dense(256)
layer.build((None, 512))
w = layer.kernel.numpy()
bound = math.sqrt(6 / (512 + 256))
print(f'range [{w.min():.4f}, {w.max():.4f}], predicted bound {bound:.4f}')
print(f'std {w.std():.4f}, predicted {bound / math.sqrt(3):.4f}')
print(f'bias all zero: {bool((layer.bias.numpy() == 0).all())}')

The weights follow the documented rule. A network with one or two hidden
layers tolerates such a default; a deep one may not. When should we override
it? Two reasons concern the scale of the signal: a deep network without
normalization layers, where the factor of each layer compounds across depth,
and an architecture whose structure the per-layer rule ignores, such as the
residual networks of this section. Two more are practical: reproducing a paper
whose results depend on its initialization, and parameters we create by hand.

In the last case Keras supplies a default too: `add_weight` falls back to
Glorot uniform unless we pass an `initializer`, a sensible scale for a weight
matrix and an unsuitable one for a parameter with another role.

## Applying Initializers

The defaults apply one fixed rule to each type of layer. To use He
initialization, or the residual scaling later in this section, we have to
apply a rule of our own, to a whole model or to selected parameters. Let us
start with the whole model: a stack of ten linear layers of width 256, each
followed by a ReLU. We leave out the biases, so that the weights alone set the
scale, and feed the stack 1,024 inputs with entries drawn from
$\mathcal{N}(0, 1)$. We measure the mean square of the output under the
default initialization, apply He initialization to every layer, and measure
again.

Every Keras layer accepts `kernel_initializer` and `bias_initializer`
arguments, objects that map a shape and a dtype to a tensor, and
`tf.keras.initializers` supplies the standard ones. For a model that already
exists, we iterate over `model.layers` and `assign` a fresh draw to each
kernel, which overwrites the variable in place, so that it remains the
variable that the model and any optimizer track. There is one trap. An
initializer object created without a seed returns the same values every time
it is called, so two layers of equal shape that share one such object start
with identical weights, and nothing warns us. The function below therefore
receives an initializer *class* and creates one object per layer. Glorot
uniform gives a square layer variance $1/n_\textrm{in}$, half of He's, so each
layer with its ReLU halves the mean square, and ten layers multiply it by
$2^{-10} \approx 10^{-3}$.

In [ ]:
net = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', use_bias=False)
    for _ in range(10)])
X = tf.random.normal((1024, 256))
print(f'default: mean square of the output '
      f'{float(tf.reduce_mean(net(X) ** 2)):.2g}')

def reinit(model, initializer_class):
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            init = initializer_class()   # one object per layer
            layer.kernel.assign(init(layer.kernel.shape))

reinit(net, tf.keras.initializers.HeNormal)
print(f'He:      mean square of the output '
      f'{float(tf.reduce_mean(net(X) ** 2)):.2g}')

Under the default, the mean square of the output is orders of magnitude below
that of the input, because each layer multiplies it by the same factor below
one. After He initialization it lies within a factor of about two of 1, the
mean square of the input. That it is not 1 itself is no fault of the rule.
The factor of each layer fluctuates with its random weights, and ten such
factors multiply.


### Truncated Normals

A normal distribution gets the variance right, but its tails are unbounded.
That is harmless for one draw. At scale, a value far in the tail is a
near-certainty: among the $10^8$ weights of a large model, dozens land beyond
five standard deviations. Large language and vision models such as BERT and
the vision transformer use a *truncated normal* instead, which cuts the tails
off at a fixed multiple of the nominal standard deviation
[@Devlin.Chang.Lee.ea.2018; @Dosovitskiy.Beyer.Kolesnikov.ea.2021]. Cutting
the tails also removes variance, which some initializers restore by rescaling
and others do not.

`tf.keras.initializers.TruncatedNormal` cuts at two standard deviations: draws
outside the bound are discarded and redrawn, and no rescaling follows. The
factory behind `HeNormal` and its relatives draws a truncated normal too and
then rescales it so that the standard deviation lands on target:

In [ ]:
w = tf.keras.initializers.RandomNormal(stddev=0.02)((1000, 1000)).numpy()
print(f'normal:    std {w.std():.4f}, max weight {abs(w).max():.4f}')
w = tf.keras.initializers.TruncatedNormal(stddev=0.02)((1000, 1000)).numpy()
print(f'truncated: std {w.std():.4f}, max weight {abs(w).max():.4f}')

A million plain normal draws produce entries near $5\sigma = 0.1$; the
truncated version guarantees $|w| \leq 0.04$. Its standard deviation falls
from 0.02 to about 0.0176, the variance that the cut removed. Recipes with a
fixed standard deviation, such as BERT's, accept this smaller variance; a
recipe whose derivation needs a target variance should rescale after
truncating.

## Custom Initializers and Direct Edits

When no library routine draws what we want, we write the initializer
ourselves, and sometimes we set values by hand. Both rest on the interface of
an initializer:

In Keras, an initializer is an object that maps `(shape, dtype)` to a tensor:
subclass `tf.keras.initializers.Initializer` and implement `__call__`. Layers
then accept an instance through their initializer arguments.

### A Rule with a Gain of Our Choice

Our example is the rule of the introduction with the gain left open: a normal
distribution with standard deviation $g/\sqrt{n_\textrm{in}}$ for any $g$. He
initialization is the case $g = \sqrt{2}$. The residual networks at the end of
this section need the rule for the last layer of each body with three more
gains, 1, $1/\sqrt{N}$ for a stack of $N$ blocks, and 0, so we write it once,
as a function of the gain.

The subclass stores the gain. A Keras kernel has shape
$(n_\textrm{in}, n_\textrm{out})$:

In [ ]:
class ScaledNormal(tf.keras.initializers.Initializer):
    def __init__(self, gain):
        self.gain = gain

    def __call__(self, shape, dtype=None):
        std = self.gain / math.sqrt(shape[0])
        return std * tf.random.normal(shape, dtype=dtype or tf.float32)

### Direct Edits

An initializer sets every entry of a parameter by one rule. We can also change
values by hand, after initialization or after training. Halving the weights
of the last layer of the stack above halves its outputs before the ReLU, and
the ReLU passes the factor on, so the mean square of the output should fall
by a factor of four.

No guard is needed, because `assign` writes to the variable without recording
anything on a `GradientTape`:

In [ ]:
before = float(tf.reduce_mean(net(X) ** 2))
w = net.layers[-1].kernel
w.assign(0.5 * w)
print(f'{before:.2g} before, {float(tf.reduce_mean(net(X) ** 2)):.2g} after')

The second number is a quarter of the first.

## Initialization of Residual Networks

### Variance Growth across Blocks

The introduction claimed that a residual network explodes although every
layer follows the rule. Let us do the calculation. A residual block computes
$\mathbf{x}_{k+1} = \mathbf{x}_k + f_k(\mathbf{x}_k)$, where the body $f_k$
is a small network. We use the body of the blocks of
that section, widened fourfold as in many residual
networks: a linear layer from $d$ to $4d$ features, a ReLU, and a linear
layer from $4d$ back to $d$, the *output projection* of the body. The first
layer gets He initialization, gain $\sqrt{2}$, because a ReLU follows it. The output projection gets standard deviation $g/\sqrt{4d}$
with a gain $g$ that we leave open for now; $g = 1$ is the choice of
that section for a layer that feeds no nonlinearity. All
biases start at zero.

How does the mean square of the activations change across one block? Write
$m_k$ for the mean square of the entries of $\mathbf{x}_k$. The first layer
doubles it and the ReLU halves it again (that section), so
together they pass on $m_k$. The output projection multiplies it by $4d \cdot
g^2/(4d) = g^2$, so the output of the body has mean square $g^2 m_k$. So far
this is routine. The step that needs an argument is the sum. The output of
the body is
uncorrelated with $\mathbf{x}_k$, because the weights of the output projection
have mean zero and are independent of $\mathbf{x}_k$. The mean square of the
sum is therefore the sum of the mean squares,

$$m_{k+1} = m_k + g^2 m_k = (1 + g^2)\, m_k.$$

Each block multiplies the mean square by $1 + g^2$, a factor above one for
every $g \neq 0$. With $g = 1$ it doubles, and after $N$ blocks the standard
deviation has grown by $2^{N/2}$: a factor of 2 after 2 blocks, 16 after 8,
and $65{,}536$ after 32. This resolves the puzzle of the introduction. Every
layer was initialized by the rule for one layer, and the network as a whole
explodes all the same, because that rule knows nothing of the skip connection,
which adds the input back in every block.

### Scaling Down the Output Projection

The factor $1 + g^2$ per block compounds to $(1 + g^2)^N$ after $N$ blocks. To
keep the product bounded as the network gets deeper, the gain must shrink with
$N$. How fast? The choice $g^2 = 1/N$, that is, scaling the output projection
by $1/\sqrt{N}$, turns out to be just enough:

$$m_N = \left(1 + \frac{1}{N}\right)^N m_0 \leq e\, m_0,$$

since $(1 + 1/N)^N$ increases toward $e$. The standard deviation then grows by
at most $\sqrt{e} \approx 1.65$, however deep the network: by 1.5 after 2
blocks, 1.60 after 8, and 1.64 after 32. The rule is in practical use. The
language model GPT-2 [@Radford.Wu.Child.ea.2019] scales the output
projection of every residual body (often called a residual *branch*) by
$1/\sqrt{N}$, where $N$ counts the bodies.

### Zero Output Projections

The extreme choice is $g = 0$. At first sight this looks reckless:
zero-initializing *every* weight is fatal, since all units of a layer then
stay identical forever (that section). Zero-initializing
only the output projection is a different matter. The body then contributes
nothing, each block is the identity map, and the network starts as a shallow
function whose depth comes into play as training proceeds. Symmetry is not a
problem, because the body's first layer keeps its random weights. On the
first backward pass, the output projection receives a nonzero gradient,
because its input is nonzero, while the first layer of the body receives
zero gradient through the zero projection. Once the projection has moved off
zero, gradient reaches the whole body.

### Measuring Variance across Depth

Three predictions as definite as these ought to be checked by measurement.
`WideResidualBlock` is the residual block of that section
with its body widened fourfold and an explicit initialization: the gain of
the output projection is a constructor argument, and both layers draw their
weights from the scaled normal initializer above, the first with gain
$\sqrt{2}$ and the output projection with that gain:

In [ ]:
class WideResidualBlock(tf.keras.layers.Layer):
    def __init__(self, d, out_gain, **kwargs):
        super().__init__(**kwargs)
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(
                4 * d, activation='relu',
                kernel_initializer=ScaledNormal(math.sqrt(2))),
            tf.keras.layers.Dense(
                d, kernel_initializer=ScaledNormal(out_gain))])

    def call(self, X):
        return X + self.body(X)

The function below stacks `num_blocks` blocks of width 64, feeds them 256
examples with entries drawn from $\mathcal{N}(0, 1)$, and records the
standard deviation of the activations at the input and after every block:

In [ ]:
def std_across_depth(num_blocks, out_gain):
    net = tf.keras.Sequential([WideResidualBlock(64, out_gain)
                               for _ in range(num_blocks)])
    X = tf.random.normal((256, 64))
    stds = [float(tf.math.reduce_std(X))]
    for block in net.layers:
        X = block(X)
        stds.append(float(tf.math.reduce_std(X)))
    return stds

The three choices of gain, unscaled, scaled, and zero, set the gain of the
output projection to 1, to $1/\sqrt{N}$, and to 0. We print the standard
deviation after the last block for $N = 2, 8, 32$ and plot it after every
block for $N = 32$:

In [ ]:
gains = {'unscaled': lambda n: 1.0, 'scaled': lambda n: n ** -0.5,
         'zero': lambda n: 0.0}
profiles = {n: [std_across_depth(n, gain(n)) for gain in gains.values()]
            for n in (2, 8, 32)}
print(f'{"N":>3}' + ''.join(f'{name:>10}' for name in gains))
for n, stds in profiles.items():
    print(f'{n:>3}' + ''.join(f'{s[-1]:>10.3g}' for s in stds))
d2l.plot(list(range(33)), profiles[32], 'blocks', 'std of activations',
         legend=list(gains), yscale='log', figsize=(4.5, 3))

The measurements follow the predictions. The unscaled column grows by about
$\sqrt{2}$ per block, a straight line on the logarithmic axis. After 32 blocks
it lands between about 20,000 and 200,000, in two runs out of three below the
predicted 65,536. The spread is no fault of the measurement. The prediction is
the square root of the expected mean square; the factor of each block
fluctuates with its random weights, the fluctuations compound, and the typical
run lies below the square root of the average. We have not trained this stack,
and with initial activations in the tens of thousands there is little reason
to try. The scaled column stays between about 1.4 and 1.9 at every depth, near
the predicted 1.5 to 1.65, and the zero column keeps the standard deviation of
the input, 1, since every block is the identity.

A forward pass of this kind catches an unusable initialization before training
begins. It cannot rank the two usable ones. Both keep the activations at
scale, and they differ only in how training starts: with every body active
under the $1/\sqrt{N}$ scaling, and with none under the zero projection.

Initialization only sets the starting point. Once training moves the weights,
nothing in this section keeps the activations at scale. That is the task of
the normalization layers of that section.

## Summary

A framework initializes its layers by default, with a fixed rule that is not
He initialization, so a deep model without normalization layers needs an
explicit rule. Initializers are applied over the tree of a model: as
constructor arguments, as a function applied to every submodule, or by
assigning new values to selected parameters. Truncated normals bound the
largest weights, and an initializer of our own takes a few lines of code.
The per-layer rules of
that section are not enough for a residual network: each
block adds its body to its input and multiplies the mean square of the
activations by $1 + g^2$, where $g$ is the gain of the body's output
projection. With $g = 1$ the standard deviation grows by $\sqrt{2}$ per block;
scaling the output projection by $1/\sqrt{N}$ bounds the growth by $\sqrt{e}$
at any depth $N$, and a zero output projection makes every block start as the
identity.

## Exercises

1. **Truncated normal.** For a normal distribution truncated at
   $\pm 2\sigma$: what fraction of draws does the cut discard, and by how
   much does it shrink the standard deviation? Neither quantity has an
   elementary closed form; express both in terms of the standard normal
   density and distribution function and evaluate them using tabulated
   values. Verify the standard deviation against the printed output of the
   demonstration in that section, and the discarded
   fraction by counting the entries of its plain normal draw that lie
   beyond $\pm 2\sigma$.

2. **Zero initialization.** Zero-initialize *both* layers of every body,
   weights and biases, instead of just the output projection. The forward
   pass still returns the input. Which parameters receive a nonzero gradient
   on the first backward pass, and on any later one? Is this the same failure
   as in the symmetry argument of that section?

3. [code] **Normalized inputs.** Many residual networks normalize the
   input of each body: the body reads $\mathbf{x}_k / r_k$, where $r_k$
   is the root mean square of the entries of $\mathbf{x}_k$ for each
   example, instead of $\mathbf{x}_k$.
    1. Show that the body output then has mean square $g^2$, whatever the
       scale of $\mathbf{x}_k$, and that with $m_0 = 1$ the mean square
       after $N$ blocks is $1 + N g^2$. How does the growth with depth
       compare with the unnormalized blocks of this section?
    1. Which gain keeps the mean square at the end of the stack equal to 2
       for every $N$? Compare it with the scaling of GPT-2.
    1. Add the normalization to `WideResidualBlock`, measure the standard
       deviation across depth for $g = 1$ and $g = 1/\sqrt{N}$ at $N = 32$,
       and compare with your predictions and with the
       measured plot in that section.

4. [code] **Kaiming initialization.** Implement the Kaiming (He)
   initializers of that section, uniform and normal, with an
   explicit `nonlinearity` argument that switches the gain between `'relu'`
   and `'linear'`.
    1. Check both on the weight of a layer with 512 inputs and 256
       outputs by mean, standard deviation, and range: the normal
       initializer against the scaled normal initializer of this
       section, and the uniform initializer against its
       predicted bound and standard deviation. Explain any difference in
       range.
    1. Apply your normal initializer to the stack of
       that section with the `'relu'` gain on
       both layers of every block. Predict the factor by which each block
       multiplies the mean square, and check it against a measured curve
       added to the plot of that section.

    *Adapted from CMU 10-714, [Homework 2](https://github.com/dlsyscourse/hw2)
    and [Homework 4](https://github.com/dlsyscourse/hw4).*

5. [code] **Gradients across depth.** ● The backward pass through a block
   multiplies the gradient by the transposed Jacobian of
   $\mathbf{x} \mapsto \mathbf{x} + f(\mathbf{x})$.
    1. Repeat the argument of this section for the backward pass. Show that
       the body multiplies the mean square of the gradient by $g^2$,
       although the backward pass sums over the outputs of each layer
       rather than its inputs, so that each block again multiplies it by
       $1 + g^2$.
    1. Verify the prediction for the three choices of gain at $N = 32$:
       compute the gradient of the sum of the outputs of the stack with
       respect to its input, and compare its mean square with that of the
       gradient at the output.

6. [code] **Data-driven calibration.** ● Initialize the stack of
   that section with the unscaled choice of
   gain, then calibrate it block by block on one minibatch of
   unit-variance inputs.
   For each block in turn, run the minibatch forward to that block,
   measure the standard deviation of its body output
   $f_k(\mathbf{x}_k)$, and rescale the block's output projection so that
   this standard deviation becomes $1/\sqrt{N}$. This adapts the
   layer-sequential unit-variance initialization of
   @Mishkin.Matas.2016 to a residual stack.
    1. Explain why the blocks must be calibrated in order rather than all
       at once from a single forward pass.
    1. Explain why the calibration targets the body output rather than
       the block output $\mathbf{x}_k + f_k(\mathbf{x}_k)$.
    1. Predict the standard deviation at the end of the calibrated stack.
       Plot the calibrated depth-versus-standard-deviation curve next to
       that of the scaled choice of gain, and explain the difference.


[Discussions](https://d2l.discourse.group/t/36754)